# 🎓 Capstone Project: Forecasting Temperature with ANN & LSTM
**Dataset:** Jena Climate 2009–2016 (Max Planck Institute, Germany) — weather readings every 10 minutes, 14 features, ~420,000 rows
**Total time:** 6 hours | **Total marks:** 100 | **Framework:** TensorFlow / Keras

---
### 📌 Problem Statement
You are a data scientist at a weather-analytics company. Your manager wants to know:
> *"Given the past **5 days (120 hours)** of weather readings, can we predict the **temperature 24 hours from now**, and is a sequence model (LSTM) actually better than a simple feed-forward network (ANN)?"*

### 🕒 Time Plan
| Part | Topic | Time | Marks |
|---|---|---|---|
| 0 | Setup & Data Loading | 20 min | 0 |
| 1 | Exploratory Data Analysis | 60 min | 20 |
| 2 | Preprocessing & Windowing | 40 min | 10 |
| 3 | Baselines | 15 min | 5 |
| 4 | ANN Model | 50 min | 15 |
| 5 | LSTM Model | 70 min | 20 |
| 6 | Optimizer & Learning-Rate Study | 45 min | 10 |
| 7 | Final Evaluation & Error Analysis | 35 min | 10 |
| 8 | Report / Conclusion | 25 min | 10 |

### ✅ Rules
- Answer in the cells marked `# YOUR CODE HERE`. Write your **observations in the markdown cells** — code without interpretation gets half marks.
- Set random seeds (given below) so results are reproducible.
- **Never** shuffle the data before splitting — this is a time series (data leakage = heavy penalty).
- Keep a **results table** (Part 7) — every model must be compared on the same test set.

## Part 0 — Setup & Data Loading 

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import os, random, time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
sns.set_style("whitegrid")
print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Download dataset (works in Colab / Kaggle / local with internet)
zip_path = keras.utils.get_file(
    origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip",
    fname="jena_climate_2009_2016.csv.zip", extract=True)
csv_path = zip_path.replace(".zip", "") if zip_path.endswith(".zip") else zip_path
if not os.path.exists(csv_path):
    csv_path = os.path.join(os.path.dirname(zip_path), "jena_climate_2009_2016.csv")
df = pd.read_csv(csv_path)
df.head()

> 💡 **Alternative (Kaggle):** search Kaggle for *"Jena Climate"* and load the CSV from `/kaggle/input/...` if internet is blocked.

## Part 1 — Exploratory Data Analysis ( 20 marks)

**Q1.1 (3 marks) – Data understanding.** Print shape, dtypes, `describe()`, number of missing values and duplicate rows. Convert `Date Time` to datetime (format `%d.%m.%Y %H:%M:%S`) and set it as index. What is the time range and sampling interval? Is the interval consistent everywhere?

In [ ]:
print("Shape:", df.shape)
print("\\nData types:"); print(df.dtypes)
print("\\nDescribe:"); display(df.describe(include="all"))
print("\\nMissing values:"); display(df.isna().sum())
print("Duplicates:", df.duplicated().sum())
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
df = df.sort_values("Date Time").set_index("Date Time")
intervals = df.index.to_series().diff().dropna()
print("Range:", df.index.min(), "to", df.index.max())
print("Mode interval:", intervals.mode().iloc[0], "| Consistent:", intervals.nunique()==1)

**Observations:** The dataset contains continuous weather measurements with numeric sensor features and a timestamp. The timestamp is parsed and used as the index so the series can be checked for regular sampling and later split chronologically. The printed shape, dtypes, missing-value counts, duplicate count, time range, and interval checks provide the data-quality summary; the notebook reports the exact values when executed.

**Q1.2 (3 marks) – Data quality.** Look at the `wv (m/s)` and `max. wv (m/s)` columns. Do you notice impossible values (e.g., -9999)? Find them, explain why they are errors, and decide how to fix them (replace with 0? NaN + interpolation?). **Justify your choice.** Apply the fix to `df`.

In [ ]:
wind_cols=["wv (m/s)","max. wv (m/s)"]
for col in wind_cols:
    invalid=df[col]<0
    print(col, "invalid:", invalid.sum())
    df.loc[invalid,col]=np.nan
df[wind_cols]=df[wind_cols].interpolate(method="time").bfill().ffill()
print("Remaining missing:", df[wind_cols].isna().sum().sum())

**Observations / justification:** Negative wind-speed values such as `-9999` are sentinel codes, not physically meaningful wind speeds. They are converted to missing values and interpolated using time, with edge gaps filled from the nearest available observation. This preserves the hourly time series and avoids treating an error code as an extreme wind event.

**Q1.3 (3 marks) – Resampling.** 10-minute data is too fine for a 5-day window (720 steps!). Resample to **hourly** data (take every 6th row or use `.resample('1H').mean()`). Report the new shape. Why does this help the model and the training time?

In [ ]:
hourly=df.resample("1h").mean()
print("Hourly shape:",hourly.shape,"Interval:",hourly.index.to_series().diff().dropna().mode().iloc[0])
display(hourly.head())

**Answer:** Hourly resampling reduces six 10-minute observations to one mean per hour. This reduces the number of time steps and noise, makes the 120-step input represent five days rather than 50 hours, and reduces memory and training time. The exact resulting row count and feature count are printed above.

**Q1.4 (4 marks) – Target & distributions.** (a) Plot the temperature `T (degC)` over the whole period. (b) Plot histograms/KDEs of **all** features in a grid. (c) Draw box plots to detect outliers. Which features are skewed? Which have outliers that are *real* (weather extremes) vs *errors*?

In [ ]:
fig,ax=plt.subplots(figsize=(16,4)); hourly["T (degC)"].plot(ax=ax,linewidth=.6)
ax.set_title("Hourly Temperature"); ax.set_ylabel("°C"); plt.tight_layout(); plt.show()
numeric_cols=hourly.select_dtypes(include=np.number).columns
ncols=3; nrows=int(np.ceil(len(numeric_cols)/ncols))
fig,axes=plt.subplots(nrows,ncols,figsize=(16,3.5*nrows)); axes=axes.ravel()
for ax,col in zip(axes,numeric_cols): sns.histplot(hourly[col].dropna(),kde=True,ax=ax); ax.set_title(col)
for ax in axes[len(numeric_cols):]: ax.remove()
plt.tight_layout(); plt.show()
fig,axes=plt.subplots(nrows,ncols,figsize=(16,3.5*nrows)); axes=axes.ravel()
for ax,col in zip(axes,numeric_cols): sns.boxplot(x=hourly[col],ax=ax); ax.set_title(col)
for ax in axes[len(numeric_cols):]: ax.remove()
plt.tight_layout(); plt.show()

**Observations:** Temperature varies smoothly over time with seasonal changes and short-term fluctuations. Histograms reveal each feature's distribution and skew; boxplots flag unusual values. Extreme weather readings can be genuine, while sentinel values or physically impossible sensor readings are errors. Outlier plots alone cannot establish that a point is erroneous, so suspicious values should be checked against domain knowledge and neighbouring timestamps.

**Q1.5 (4 marks) – Time patterns.** (a) Plot average temperature **by month** and **by hour of day**. (b) Zoom into any 2-week window and plot it. (c) Perform a seasonal decomposition (`statsmodels.tsa.seasonal.seasonal_decompose`, period=24 on a one-month slice) or plot the autocorrelation (`pandas.plotting.autocorrelation_plot`). What repeating cycles do you find? What does this tell you about how long a window the model needs?

In [ ]:
monthly_mean=hourly.groupby(hourly.index.month)["T (degC)"].mean()
hourly_mean=hourly.groupby(hourly.index.hour)["T (degC)"].mean()
fig,axes=plt.subplots(1,2,figsize=(13,4))
monthly_mean.plot(marker="o",ax=axes[0]); axes[0].set_title("Mean Temp by Month"); axes[0].set_xlabel("Month")
hourly_mean.plot(marker="o",ax=axes[1]); axes[1].set_title("Mean Temp by Hour"); axes[1].set_xlabel("Hour")
plt.tight_layout(); plt.show()
two_weeks=hourly.loc["2016-01-01":"2016-01-14","T (degC)"]
if two_weeks.empty: two_weeks=hourly.iloc[:14*24]["T (degC)"]
two_weeks.plot(figsize=(14,4),title="Two-week Temperature"); plt.ylabel("°C"); plt.tight_layout(); plt.show()
try:
 from statsmodels.tsa.seasonal import seasonal_decompose
 one_month=hourly["T (degC)"].dropna().iloc[:30*24]
 seasonal_decompose(one_month,model="additive",period=24).plot(); plt.tight_layout(); plt.show()
except ImportError:
 from pandas.plotting import autocorrelation_plot
 autocorrelation_plot(hourly["T (degC)"].dropna().iloc[:30*24]); plt.title("Autocorrelation"); plt.tight_layout(); plt.show()

**Observations:** Temperature shows an annual cycle (warmer and colder months) and a daily cycle (heating during the day and cooling overnight). The two-week plot shows short-term changes, while the 24-hour decomposition separates daily seasonal structure. A 120-hour input window covers five daily cycles, providing recent context for daily weather patterns; longer-term seasonal context is represented partly by the cyclical month features.

**Q1.6 (3 marks) – Relationships.** Plot a correlation heatmap. Which features are almost perfectly correlated with temperature (e.g., `Tpot (K)`)? Is it **valid** to keep a feature that is basically the target in a different unit? Which features would you drop and why? Also create **cyclical time features**: `hour_sin, hour_cos, month_sin, month_cos` (hint: `np.sin(2*np.pi*hour/24)`) and explain why sin/cos is better than raw hour numbers.

In [ ]:
corr=hourly.corr(numeric_only=True)
plt.figure(figsize=(13,10)); sns.heatmap(corr,cmap="coolwarm",center=0); plt.title("Feature Correlation"); plt.tight_layout(); plt.show()
display(corr["T (degC)"].sort_values(ascending=False))
model_df=hourly.drop(columns=["Tpot (K)"],errors="ignore").copy()
hour,month=model_df.index.hour,model_df.index.month
model_df["hour_sin"]=np.sin(2*np.pi*hour/24); model_df["hour_cos"]=np.cos(2*np.pi*hour/24)
model_df["month_sin"]=np.sin(2*np.pi*(month-1)/12); model_df["month_cos"]=np.cos(2*np.pi*(month-1)/12)
print("Features:",list(model_df.columns))

**Observations:** `Tpot (K)` is strongly correlated with `T (degC)` because it is essentially the same temperature expressed in Kelvin. Keeping it would leak the target almost directly and artificially improve performance, so it is dropped. Sine/cosine encodings map hour and month onto circular coordinates: 23:00 and 00:00 (or December and January) become close, unlike raw integer encodings that imply an artificial boundary.

## Part 2 — Preprocessing & Windowing (10 marks)

**Q2.1 (2 marks) – Chronological split.** Split the hourly data **70% train / 15% validation / 15% test** *in time order* (no shuffling). Print the date ranges of each split.

In [ ]:
n=len(model_df); train_end=int(n*.70); val_end=int(n*.85)
train_df=model_df.iloc[:train_end].copy(); val_df=model_df.iloc[train_end:val_end].copy(); test_df=model_df.iloc[val_end:].copy()
for name,part in [("Train",train_df),("Validation",val_df),("Test",test_df)]:
 print(name,part.index.min(),"to",part.index.max(),"rows:",len(part))

**Q2.2 (2 marks) – Scaling without leakage.** Fit `StandardScaler` on the **train set only**, then transform val and test. Explain in one line what leakage would happen if you fitted it on the whole dataset. Keep a separate scaler (or the mean/std) for the target so you can inverse-transform predictions to °C.

In [ ]:
feature_cols=list(model_df.columns); target_col="T (degC)"; target_idx=feature_cols.index(target_col)
feature_scaler=StandardScaler().fit(train_df[feature_cols])
train_scaled=feature_scaler.transform(train_df[feature_cols]); val_scaled=feature_scaler.transform(val_df[feature_cols]); test_scaled=feature_scaler.transform(test_df[feature_cols])
target_scaler=StandardScaler().fit(train_df[[target_col]])
y_train_raw=train_df[target_col].to_numpy(); y_val_raw=val_df[target_col].to_numpy(); y_test_raw=test_df[target_col].to_numpy()
print("Target scaler mean/std:",target_scaler.mean_[0],target_scaler.scale_[0])

**Why train-only?** Fitting the scaler on all rows leaks information about the validation and test distributions into training; fit only on training data, then reuse those fitted parameters to transform the later splits.

**Q2.3 (4 marks) – Build windows.** Write a function `make_windows(data, target_idx, past=120, horizon=24)` that returns:
- `X` of shape `(samples, 120, n_features)` — the past 120 hours
- `y` of shape `(samples,)` — the temperature **24 hours after** the last hour of the window

Apply it to train/val/test. Print all shapes. (You may also use `tf.keras.utils.timeseries_dataset_from_array`.) **Draw/describe one window by hand to prove your indexing is not off by one.**

In [ ]:
PAST,HORIZON=120,24
def make_windows(data,target_idx,past=PAST,horizon=HORIZON):
 data=np.asarray(data,dtype=np.float32); X,y=[],[]
 n_samples=len(data)-past-horizon+1
 for start in range(max(0,n_samples)):
  X.append(data[start:start+past])
  y.append(data[start+past+horizon-1,target_idx])
 return np.asarray(X,dtype=np.float32),np.asarray(y,dtype=np.float32)
X_train,y_train=make_windows(train_scaled,target_idx); X_val,y_val=make_windows(val_scaled,target_idx); X_test,y_test=make_windows(test_scaled,target_idx)
print("Train:",X_train.shape,y_train.shape,"Validation:",X_val.shape,y_val.shape,"Test:",X_test.shape,y_test.shape)
print("First input:",train_df.index[0],"to",train_df.index[PAST-1],"Target:",train_df.index[PAST+HORIZON-1],"Scaled:",y_train[0])

**Window indexing check:** The input uses rows `start` through `start+119` (120 hours). The target is at `start+143`, which is 24 hours after the final input row. The printed first-window timestamps verify this.

**Q2.4 (2 marks) – tf.data pipeline.** Wrap the arrays in `tf.data.Dataset` with `batch(64)` and `prefetch(tf.data.AUTOTUNE)`. Shuffle **only the training windows** (shuffling windows is fine; shuffling the raw series is not). Why is that distinction valid?

**Answer:** Shuffling completed training windows is valid because each window retains its own chronological sequence and label. Only the order of independent training examples changes; validation and test windows remain ordered.

In [ ]:
train_ds=tf.data.Dataset.from_tensor_slices((X_train,y_train)).shuffle(min(len(X_train),10000),seed=SEED,reshuffle_each_iteration=True).batch(64).prefetch(tf.data.AUTOTUNE)
val_ds=tf.data.Dataset.from_tensor_slices((X_val,y_val)).batch(64).prefetch(tf.data.AUTOTUNE)
test_ds=tf.data.Dataset.from_tensor_slices((X_test,y_test)).batch(64).prefetch(tf.data.AUTOTUNE)

## Part 3 — Baselines (5 marks)
A deep model is only useful if it beats something simple.

**Q3.1 (3 marks)** Compute test-set **MAE and RMSE (in °C)** for two baselines:
1. *Persistence*: predict that temperature 24 h from now = temperature right now.
2. *Linear Regression* (or Ridge) on the flattened window `(samples, 120*features)`.

**Q3.2 (2 marks)** Why is persistence a surprisingly strong baseline for weather? Which one should the neural networks beat?

In [ ]:
def evaluate(y_true,y_pred,name=""):
 y_true=np.asarray(y_true).reshape(-1); y_pred=np.asarray(y_pred).reshape(-1)
 mae=mean_absolute_error(y_true,y_pred); rmse=np.sqrt(mean_squared_error(y_true,y_pred)); r2=r2_score(y_true,y_pred)
 print(f"{name:25s} MAE={mae:.3f} RMSE={rmse:.3f} R2={r2:.3f}")
 return {"model":name,"MAE":mae,"RMSE":rmse,"R2":r2}
results=[]
test_target_raw=test_df[target_col].iloc[PAST+HORIZON-1:].to_numpy()
persistence_scaled=X_test[:,-1,target_idx]
persistence_pred=target_scaler.inverse_transform(persistence_scaled.reshape(-1,1)).ravel()
results.append(evaluate(test_target_raw,persistence_pred,"Persistence"))
from sklearn.linear_model import Ridge
ridge=Ridge(alpha=1.0, solver='lsqr').fit(X_train.reshape(len(X_train),-1),y_train)
ridge_pred_scaled=ridge.predict(X_test.reshape(len(X_test),-1))
ridge_pred=target_scaler.inverse_transform(ridge_pred_scaled.reshape(-1,1)).ravel()
results.append(evaluate(test_target_raw,ridge_pred,"Linear Regression (Ridge)"))

**Answer:** Persistence is a strong baseline because temperature usually changes gradually and is autocorrelated; the value now is often close to the value one day later. Ridge regression provides a linear benchmark using the full input window. ANN, LSTM, and the GRU variant should be compared against both baselines on the same chronological test targets, using MAE and RMSE in °C and R².

## Part 4 — ANN (Feed-Forward) Model (15 marks)

**Q4.1 (4 marks) – Architecture.** Build a function `build_ann(input_shape, units=(128,64,32), dropout=0.2, l2=1e-4)` with `Flatten` → `Dense` layers (ReLU, with `BatchNormalization` optional) → `Dropout` → `Dense(1)` output (linear). Print `model.summary()`. Explain: why linear activation at the output? Why MSE/Huber loss for regression?

In [ ]:
def build_ann(input_shape,units=(128,64,32),dropout=.2,l2=1e-4):
 model=keras.Sequential(name="ANN"); model.add(layers.Input(shape=input_shape)); model.add(layers.Flatten())
 for width in units:
  model.add(layers.Dense(width,activation="relu",kernel_regularizer=keras.regularizers.l2(l2)))
  model.add(layers.BatchNormalization()); model.add(layers.Dropout(dropout))
 model.add(layers.Dense(1,activation="linear"))
 model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),loss="mse",metrics=["mae"])
 return model
ann_model=build_ann(X_train.shape[1:]); ann_model.summary()

**Answer:** The final Dense output is linear because regression predicts a continuous temperature and should not be constrained to a bounded class probability. MSE penalizes larger errors more strongly and is differentiable; Huber loss is an alternative that is quadratic for small errors and less sensitive to large residuals/outliers.

**Q4.2 (5 marks) – Training with callbacks.** Compile with `Adam(learning_rate=1e-3)`, loss `mse`, metric `mae`. Train for up to **100 epochs** using **all three** callbacks:
- `EarlyStopping()`
- `ModelCheckpoint()`
- `ReduceLROnPlateau()`

Answer: At which epoch did training stop? Which epoch's weights were saved? What does `restore_best_weights=True` do and what happens if you set it to `False`?

In [ ]:
ann_callbacks=[callbacks.EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True,verbose=1),callbacks.ModelCheckpoint("best_ann.keras",monitor="val_loss",save_best_only=True,verbose=1),callbacks.ReduceLROnPlateau(monitor="val_loss",factor=.5,patience=5,min_lr=1e-6,verbose=1)]
ann_start=time.time(); ann_history=ann_model.fit(train_ds,validation_data=val_ds,epochs=100,callbacks=ann_callbacks,verbose=1); ann_training_time=time.time()-ann_start
ann_best_epoch=int(np.argmin(ann_history.history["val_loss"])+1); ann_epoch_times=ann_training_time/len(ann_history.history["loss"])
print(f"ANN stopped at {len(ann_history.history['loss'])}; best epoch {ann_best_epoch}; approx sec/epoch {ann_epoch_times:.2f}")

**Answer:** Training stops when validation loss fails to improve for the configured patience, or when 100 epochs are reached. `ModelCheckpoint(save_best_only=True)` saves the weights from the lowest validation-loss epoch. `restore_best_weights=True` restores those best weights in memory at the end; setting it to `False` leaves the weights from the final epoch (although the checkpoint file still contains the best model). The notebook prints the actual stopping and best epochs.

**Q4.3 (3 marks) – Learning curves.** Plot training vs. validation loss (and MAE). Diagnose: *underfit, good fit or overfit?* Mark the epoch where the validation loss is lowest with a vertical line.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].plot(ann_history.history["loss"],label="Train"); axes[0].plot(ann_history.history["val_loss"],label="Validation"); axes[0].axvline(np.argmin(ann_history.history["val_loss"]),ls="--",label="Best epoch"); axes[0].set_title("ANN Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(ann_history.history["mae"],label="Train MAE"); axes[1].plot(ann_history.history["val_mae"],label="Validation MAE"); axes[1].set_title("ANN MAE (scaled)"); axes[1].legend()
plt.tight_layout(); plt.show()

**Diagnosis:** Use the plotted curves to decide: if both training and validation loss remain high, the model is underfitting; if training improves but validation worsens and diverges, it is overfitting; if both improve and stay close with validation loss stabilizing, it is a reasonable fit. The vertical marker identifies the lowest validation-loss epoch.

**Q4.4 (3 marks) – Load & evaluate.** Load `best_ann.keras` with `keras.models.load_model`, predict on the test set, **inverse-transform to °C**, and compute MAE / RMSE / R². Append to `results`. Plot predicted vs actual for the first 300 test samples.

In [ ]:
ann_best=keras.models.load_model("best_ann.keras")
ann_pred_scaled=ann_best.predict(X_test,batch_size=64,verbose=0).ravel()
ann_pred=target_scaler.inverse_transform(ann_pred_scaled.reshape(-1,1)).ravel()
results.append(evaluate(test_target_raw,ann_pred,"ANN")); ann_test_metrics=results[-1]
plt.figure(figsize=(14,4)); plt.plot(test_target_raw[:300],label="Actual"); plt.plot(ann_pred[:300],label="ANN"); plt.title("ANN Actual vs Predicted (first 300)"); plt.ylabel("°C"); plt.legend(); plt.tight_layout(); plt.show()

## Part 5 — LSTM Model (20 marks)

**Q5.1 (4 marks) – Architecture.** Build `build_lstm(input_shape, units=(64,32), dropout=0.2, recurrent_dropout=0.0)` with stacked `LSTM` layers (`return_sequences=True` for all but the last), `Dropout`, and a `Dense(1)` output. Print the summary.
Explain: (a) why `return_sequences=True` for the first layer? (b) why is `recurrent_dropout>0` slow on GPU? (c) Compare the number of parameters with your ANN.

In [ ]:
def build_lstm(input_shape,units=(64,32),dropout=.2,recurrent_dropout=0.0):
 model=keras.Sequential(name="LSTM"); model.add(layers.Input(shape=input_shape))
 for i,width in enumerate(units):
  model.add(layers.LSTM(width,return_sequences=(i<len(units)-1),recurrent_dropout=recurrent_dropout)); model.add(layers.Dropout(dropout))
 model.add(layers.Dense(1)); model.compile(optimizer=optimizers.Adam(learning_rate=1e-3,clipnorm=1.0),loss="mse",metrics=["mae"]); return model
lstm_model=build_lstm(X_train.shape[1:]); lstm_model.summary()

**Answers:** (a) `return_sequences=True` keeps the full timestep sequence so the next recurrent layer receives a sequence instead of one final vector. (b) Recurrent dropout can prevent use of some fast GPU kernels and therefore slow training. (c) Compare the parameter totals shown in the ANN and LSTM summaries; recurrent layers have gates and parameters that depend on input and hidden dimensions, so their count differs from dense layers.

**Q5.2 (5 marks) – Train.** Train the LSTM with the **same three callbacks** (checkpoint saved as `best_lstm.keras`), the same optimizer settings as the ANN, and `clipnorm=1.0` in Adam. Record the **training time per epoch**. Why is gradient clipping commonly used for RNNs? (Mention *exploding gradients*.)

In [ ]:
lstm_callbacks=[callbacks.EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True,verbose=1),callbacks.ModelCheckpoint("best_lstm.keras",monitor="val_loss",save_best_only=True,verbose=1),callbacks.ReduceLROnPlateau(monitor="val_loss",factor=.5,patience=5,min_lr=1e-6,verbose=1)]
lstm_start=time.time(); lstm_history=lstm_model.fit(train_ds,validation_data=val_ds,epochs=100,callbacks=lstm_callbacks,verbose=1); lstm_training_time=time.time()-lstm_start
lstm_best_epoch=int(np.argmin(lstm_history.history["val_loss"])+1); lstm_epoch_times=lstm_training_time/len(lstm_history.history["loss"])
print(f"LSTM stopped at {len(lstm_history.history['loss'])}; best epoch {lstm_best_epoch}; approx sec/epoch {lstm_epoch_times:.2f}")

**Answer:** Gradient clipping limits the gradient norm before the optimizer update. It is commonly used in recurrent networks to reduce the risk of exploding gradients, which can cause unstable or excessively large updates. The training cell prints total time and approximate average time per epoch; the actual time depends on hardware and runtime load.

**Q5.3 (3 marks) – Evaluate.** Same evaluation as the ANN (°C, MAE/RMSE/R²) — append to `results`. Plot learning curves and predicted vs. actual.

In [ ]:
lstm_best=keras.models.load_model("best_lstm.keras")
lstm_pred_scaled=lstm_best.predict(X_test,batch_size=64,verbose=0).ravel(); lstm_pred=target_scaler.inverse_transform(lstm_pred_scaled.reshape(-1,1)).ravel()
results.append(evaluate(test_target_raw,lstm_pred,"LSTM")); lstm_test_metrics=results[-1]
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].plot(lstm_history.history["loss"],label="Train"); axes[0].plot(lstm_history.history["val_loss"],label="Validation"); axes[0].axvline(np.argmin(lstm_history.history["val_loss"]),ls="--"); axes[0].set_title("LSTM Learning Curves"); axes[0].legend()
axes[1].plot(test_target_raw[:300],label="Actual"); axes[1].plot(lstm_pred[:300],label="LSTM"); axes[1].set_title("LSTM Actual vs Predicted"); axes[1].set_ylabel("°C"); axes[1].legend()
plt.tight_layout(); plt.show()
print("Parameter counts | ANN:",ann_best.count_params(),"LSTM:",lstm_best.count_params())
print("Average seconds/epoch | ANN:",round(ann_epoch_times,2),"LSTM:",round(lstm_epoch_times,2))

**Q5.4 (4 marks) – Variants.** Try **one** of the following and compare with the baseline LSTM (use the same callbacks):
- (a) `GRU` instead of `LSTM`
- (b) `Bidirectional(LSTM)` — *discuss whether bidirectional makes sense for forecasting*
- (c) `Conv1D` front-end followed by LSTM
- (d) Predict the full next 24 hours (multi-step output) instead of one value

Add the result to `results`.

In [ ]:
def build_gru(input_shape,units=(64,32),dropout=.2):
 model=keras.Sequential(name="GRU_variant"); model.add(layers.Input(shape=input_shape))
 for i,width in enumerate(units):
  model.add(layers.GRU(width,return_sequences=(i<len(units)-1))); model.add(layers.Dropout(dropout))
 model.add(layers.Dense(1)); model.compile(optimizer=optimizers.Adam(learning_rate=1e-3,clipnorm=1.0),loss="mse",metrics=["mae"]); return model
variant_model=build_gru(X_train.shape[1:])
variant_callbacks=[callbacks.EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True,verbose=1),callbacks.ModelCheckpoint("best_gru.keras",monitor="val_loss",save_best_only=True,verbose=1),callbacks.ReduceLROnPlateau(monitor="val_loss",factor=.5,patience=5,min_lr=1e-6,verbose=1)]
variant_start=time.time(); variant_history=variant_model.fit(train_ds,validation_data=val_ds,epochs=100,callbacks=variant_callbacks,verbose=1); variant_time=time.time()-variant_start
variant_best=keras.models.load_model("best_gru.keras"); variant_pred_scaled=variant_best.predict(X_test,batch_size=64,verbose=0).ravel()
variant_pred=target_scaler.inverse_transform(variant_pred_scaled.reshape(-1,1)).ravel()
results.append(evaluate(test_target_raw,variant_pred,"GRU variant")); variant_epoch_times=variant_time/len(variant_history.history["loss"])
print("GRU best epoch:",np.argmin(variant_history.history["val_loss"])+1,"sec/epoch:",round(variant_epoch_times,2))

**Discussion:** The GRU is a gated recurrent architecture with fewer gates than an LSTM and can be faster or more compact. Compare its test MAE/RMSE/R² with the baseline LSTM. Bidirectional recurrence is not always appropriate for forecasting because the model must not use future observations beyond the prediction cutoff; this variant uses a GRU and only the historical input window.

**Q5.5 (4 marks) – ANN vs LSTM.** Write a comparison (5–8 lines): accuracy, training time, parameters, overfitting behavior. Did the LSTM actually win? If not, give **two possible reasons** (e.g., window too long, small signal beyond seasonality, insufficient tuning).

**Comparison:** Compare ANN and LSTM using their test MAE/RMSE/R², parameter counts (`model.count_params()`), and measured training time. ANN treats the input window as a flattened vector and may learn useful relationships efficiently. LSTM models temporal order and dependencies, but it is slower and may not win when weather is highly autocorrelated, the forecast horizon is long, the available window is limited, or tuning/training is insufficient. Use the measured results rather than assuming the LSTM is better.

## Part 6 — Optimizer & Learning-Rate Study (10 marks)

**Q6.1 (6 marks)** Using the **better of your ANN/LSTM** architectures, train the *same model from the same initial weights* with each optimizer (max 40 epochs, EarlyStopping patience=6):

| Optimizer | Settings |
|---|---|
| SGD | lr=0.01 |
| SGD + Momentum | lr=0.01, momentum=0.9 |
| RMSprop | lr=1e-3 |
| Adam | lr=1e-3 |
| AdamW | lr=1e-3, weight_decay=1e-4 |

Plot **validation loss vs epoch for all optimizers on one chart** and make a table: *best val loss, epoch of best, time per epoch*.

In [ ]:
best_arch_name="LSTM" if min(lstm_history.history["val_loss"])<min(ann_history.history["val_loss"]) else "ANN"
best_arch_builder=build_lstm if best_arch_name=="LSTM" else build_ann; best_arch_units=(64,32) if best_arch_name=="LSTM" else (128,64,32)
optimizer_histories,optimizer_rows={},[]
for opt_name,opt_factory in optimizer_configs.items():
 tf.keras.utils.set_random_seed(SEED)
 trial=best_arch_builder(X_train.shape[1:],units=best_arch_units); trial.compile(optimizer=opt_factory(),loss="mse",metrics=["mae"])
 trial_callbacks=[callbacks.EarlyStopping(monitor="val_loss",patience=6,restore_best_weights=True)]
 t0=time.time(); hist=trial.fit(train_ds,validation_data=val_ds,epochs=40,callbacks=trial_callbacks,verbose=0); elapsed=time.time()-t0
 optimizer_histories[opt_name]=hist.history["val_loss"]; best_i=int(np.argmin(hist.history["val_loss"]))
 optimizer_rows.append({"Optimizer":opt_name,"Best val loss":hist.history["val_loss"][best_i],"Best epoch":best_i+1,"Approx. seconds/epoch":elapsed/len(hist.history["loss"])})
optimizer_table=pd.DataFrame(optimizer_rows).sort_values("Best val loss"); display(optimizer_table)
plt.figure(figsize=(10,5))
for name,losses in optimizer_histories.items(): plt.plot(losses,label=name)
plt.title("Validation Loss by Optimizer"); plt.xlabel("Epoch"); plt.ylabel("Validation MSE"); plt.legend(); plt.tight_layout(); plt.show()

**Q6.2 (2 marks) – Learning-rate sensitivity.** Train with Adam at learning rates `[1e-4, 1e-3, 1e-2, 1e-1]`. Plot the results. What happens at 1e-1 and at 1e-4?

In [ ]:
lr_values=[1e-4,1e-3,1e-2,1e-1]; lr_histories,lr_rows={},[]
for lr in lr_values:
 tf.keras.utils.set_random_seed(SEED); lr_model=best_arch_builder(X_train.shape[1:],units=best_arch_units)
 lr_model.compile(optimizer=optimizers.Adam(learning_rate=lr),loss="mse",metrics=["mae"])
 lr_cb=[callbacks.EarlyStopping(monitor="val_loss",patience=6,restore_best_weights=True)]
 lr_hist=lr_model.fit(train_ds,validation_data=val_ds,epochs=40,callbacks=lr_cb,verbose=0); lr_histories[lr]=lr_hist.history["val_loss"]
 bi=int(np.argmin(lr_hist.history["val_loss"])); lr_rows.append({"Learning rate":lr,"Best val loss":lr_hist.history["val_loss"][bi],"Best epoch":bi+1})
display(pd.DataFrame(lr_rows).sort_values("Best val loss"))
plt.figure(figsize=(10,5))
for lr,losses in lr_histories.items(): plt.plot(losses,label=f"Adam lr={lr:g}")
plt.title("Learning Rate Sensitivity"); plt.xlabel("Epoch"); plt.ylabel("Validation MSE"); plt.legend(); plt.tight_layout(); plt.show()

**Observations:** Lower learning rates generally make smaller updates and can converge slowly; higher rates may converge quickly but overshoot, oscillate, or diverge. Compare validation curves and best validation loss for all four values. The exact behavior depends on this dataset and the selected architecture.

**Q6.3 (2 marks) – Written explanation.** In your own words (max 6 lines): why does plain SGD converge slower than Adam here? What does *momentum* add? What do `ReduceLROnPlateau` and a learning-rate schedule (e.g., `CosineDecay`) do?

**Answer:** Plain SGD uses the same learning rate for all parameters and can take slow, noisy steps. Adam adapts update sizes using moving averages of gradients and squared gradients. Momentum accumulates a fraction of previous updates to accelerate consistent directions and reduce oscillation. `ReduceLROnPlateau` lowers the rate when a monitored metric stops improving; schedules such as cosine decay change it smoothly over training.

## Part 7 — Final Evaluation & Error Analysis (10 marks)

**Q7.1 (3 marks) – Results table.** Build a DataFrame from `results` with MAE, RMSE, R² of: Persistence, Linear Regression, ANN, LSTM, your variant. Highlight the best row. Plot a bar chart of MAE.

In [ ]:
results_df=pd.DataFrame(results).drop_duplicates(subset=["model"],keep="last").sort_values("MAE").reset_index(drop=True)
display(results_df.style.highlight_min(subset=["MAE","RMSE"],color="lightgreen").highlight_max(subset=["R2"],color="lightgreen"))
plt.figure(figsize=(10,5)); plt.bar(results_df["model"],results_df["MAE"]); plt.title("Test MAE by Model"); plt.ylabel("MAE °C"); plt.xticks(rotation=25,ha="right"); plt.tight_layout(); plt.show()
model_pred_map={"Persistence":persistence_pred,"Linear Regression (Ridge)":ridge_pred,"ANN":ann_pred,"LSTM":lstm_pred,"GRU variant":variant_pred}
best_model_name=results_df.iloc[0]["model"]; best_pred=model_pred_map[best_model_name]; print("Best test model:",best_model_name)

**Q7.2 (4 marks) – Error analysis.** For your best model: (a) plot residuals (histogram + residuals vs time), (b) compute MAE **per month** or **per season** — when does the model fail most? (c) show the 3 worst predictions and explain what happened in the weather around that time.

In [ ]:
residuals=test_target_raw-best_pred; test_target_times=test_df.index[PAST+HORIZON-1:]; residual_series=pd.Series(residuals,index=test_target_times)
fig,axes=plt.subplots(1,2,figsize=(13,4)); sns.histplot(residuals,kde=True,ax=axes[0]); axes[0].set_title("Residual Distribution"); axes[0].set_xlabel("Actual - Predicted °C")
axes[1].plot(residual_series.index,residual_series.values,lw=.7); axes[1].axhline(0,ls="--"); axes[1].set_title("Residuals over Time"); plt.tight_layout(); plt.show()
monthly_mae=residual_series.abs().resample("MS").mean(); monthly_mae.plot(figsize=(13,4),marker="o",title="Monthly Test MAE"); plt.ylabel("MAE °C"); plt.tight_layout(); plt.show()
worst_idx=np.argsort(np.abs(residuals))[-3:][::-1]
display(pd.DataFrame({"Timestamp":test_target_times[worst_idx],"Actual °C":test_target_raw[worst_idx],"Predicted °C":best_pred[worst_idx],"Absolute Error °C":np.abs(residuals[worst_idx])}).sort_values("Absolute Error °C",ascending=False))

**Findings:** Residuals (`actual − predicted`) centered near zero indicate limited systematic bias; a wide or skewed distribution signals errors or bias. Monthly MAE reveals periods with poorer performance, which may correspond to unusual weather or seasonal transitions. Inspect the timestamps and surrounding temperature pattern for the three largest errors before attributing a cause; the notebook lists the actual and predicted temperatures and their error sizes.

**Q7.3 (3 marks) – Feature importance (permutation).** For the best model, shuffle **one feature at a time** across the test windows and measure the increase in MAE. Plot the top features. Does it match your EDA intuition?

In [ ]:
baseline_mae=mean_absolute_error(test_target_raw,best_pred); importance_rows=[]; rng=np.random.default_rng(SEED)
for feature_i,feature_name in enumerate(feature_cols):
 X_perm=X_test.copy(); shuffled=X_perm[:,:,feature_i].copy().reshape(-1); rng.shuffle(shuffled); X_perm[:,:,feature_i]=shuffled.reshape(X_perm[:,:,feature_i].shape)
 if best_model_name=="ANN": pred_s=ann_best.predict(X_perm,batch_size=64,verbose=0).ravel()
 elif best_model_name=="LSTM": pred_s=lstm_best.predict(X_perm,batch_size=64,verbose=0).ravel()
 elif best_model_name=="GRU variant": pred_s=variant_best.predict(X_perm,batch_size=64,verbose=0).ravel()
 elif best_model_name=="Linear Regression (Ridge)": pred_s=ridge.predict(X_perm.reshape(len(X_perm),-1))
 else: pred_s=target_scaler.transform(X_perm[:,-1,target_idx].reshape(-1,1)).ravel()
 pred_deg=target_scaler.inverse_transform(pred_s.reshape(-1,1)).ravel(); perm_mae=mean_absolute_error(test_target_raw,pred_deg)
 importance_rows.append({"Feature":feature_name,"MAE increase":perm_mae-baseline_mae})
importance_df=pd.DataFrame(importance_rows).sort_values("MAE increase",ascending=False); display(importance_df.head(10))
importance_df.head(10).sort_values("MAE increase").plot.barh(x="Feature",y="MAE increase",figsize=(9,5),legend=False); plt.title("Top Permutation Importance"); plt.xlabel("Increase in MAE °C"); plt.tight_layout(); plt.show()

**Findings:** Permutation importance is the increase in test MAE after a feature is shuffled; larger positive increases indicate the model relies more on that feature. Correlated features can share importance, and shuffling can create unrealistic combinations, so interpret rankings as a diagnostic rather than causal evidence. Compare the ranking with the correlation and time-pattern EDA.

## Part 8 — Report & Conclusion (10 marks)

**Q8.1 (5 marks) – Executive summary** (150–200 words) for your manager: the problem, the best model, its error in °C, and whether the complexity of an LSTM is worth it.

**Q8.2 (3 marks) – Limitations & next steps** (at least 3 each): e.g., only one location, no weather-forecast inputs, trend drift across years, uncertainty estimates, etc.

**Q8.3 (2 marks) – Reproducibility.** Save your best model, the scaler (`joblib.dump`), and list the versions of the libraries you used. Restart the kernel and **Run All** — the notebook must run top to bottom without error.

**Executive Summary:**

This capstone evaluates whether hourly weather measurements from the Jena Climate dataset can forecast temperature 24 hours ahead using a five-day input window. The workflow first checks the data quality, resamples the observations to hourly frequency, removes a near-duplicate temperature feature to avoid target leakage, and adds cyclical time encodings. Persistence and Ridge regression provide simple reference baselines. A feed-forward ANN, stacked LSTM, and GRU variant are trained with validation-based early stopping, checkpointing, and learning-rate reduction. All models are evaluated on the same chronological test segment using MAE, RMSE, and R² in degrees Celsius. The best model is selected from the measured test results and validation performance; its metrics and comparison with the baselines are reported in the results table produced by the notebook. The LSTM is worthwhile only if its accuracy improvement is meaningful relative to its additional parameter count and training cost. Model complexity should be justified by measurable improvement, not assumed superiority.

**Limitations:**
- Data represents one location and may not generalize to other climates.
- No external weather forecast, satellite, or geographic inputs are used.
- Sensor noise, missing values, and long-term climate drift may affect results.

**Next steps:**
- Add more locations and additional forecast/weather variables.
- Evaluate multiple horizons and prediction intervals/uncertainty.
- Validate on later time periods and tune the model using rolling time-series evaluation.

In [ ]:
import joblib
ann_val_best=float(np.min(ann_history.history["val_loss"])); lstm_val_best=float(np.min(lstm_history.history["val_loss"])); gru_val_best=float(np.min(variant_history.history["val_loss"]))
if ann_val_best<=lstm_val_best and ann_val_best<=gru_val_best: final_model=ann_best; final_model_name="ANN"
elif lstm_val_best<=gru_val_best: final_model=lstm_best; final_model_name="LSTM"
else: final_model=variant_best; final_model_name="GRU"
final_model.save("best_weather_model.keras"); joblib.dump(feature_scaler,"feature_scaler.joblib"); joblib.dump(target_scaler,"target_scaler.joblib")
print("Saved:",final_model_name,"and both scalers.")